# 🛒 Amazon Order Profitability Prediction
## ISOM 835 — Predictive Analytics and Machine Learning
**Suffolk University | Sawyer Business School**

---
**Objective:** Predict whether an Amazon sales order will be profitable or result in a loss, using machine learning classification models.

**Dataset:** Amazon Sales Dataset (Western US, 2011–2014) — 3,203 orders, 16 features

**Target Variable:** `Profit_Status` → Binary (1 = Profitable, 0 = Loss)

---

## 📦 1. Install & Import Libraries

In [ ]:
# Install required libraries
!pip install openpyxl -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                              confusion_matrix, classification_report, roc_curve)

# Plot styling
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.facecolor'] = '#f8f9fa'

print('✅ All libraries imported successfully!')

## 📂 2. Load Dataset

In [ ]:
# Upload the file in Colab: click the folder icon on the left → upload
# Or mount Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')

from google.colab import files
uploaded = files.upload()  # Upload CHILE_Amazon_Sales_Dataset_FINAL.xlsx

# Load the Amazon_RawData sheet
df_raw = pd.read_excel('CHILE_Amazon_Sales_Dataset_FINAL.xlsx', sheet_name='Amazon_RawData')

# Load the enriched sheet with pre-engineered features
df = pd.read_excel('CHILE_Amazon_Sales_Dataset_FINAL.xlsx', sheet_name='Chap 1 - Fundamental Excel ')
df = df[['Order ID','Order Date','Ship Date','Month','Year','Shipping Days',
         'City','State','Category','Product Name','Sales ($)',
         'Quantity','Profit','Profit Margin (%)','Profit Status','Sales Size']].copy()
df.columns = ['Order_ID','Order_Date','Ship_Date','Month','Year','Shipping_Days',
              'City','State','Category','Product_Name','Sales',
              'Quantity','Profit','Profit_Margin','Profit_Status','Sales_Size']

print(f'✅ Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

## 🔍 3. Exploratory Data Analysis (EDA)

In [ ]:
# Dataset overview
print('='*50)
print('DATASET OVERVIEW')
print('='*50)
print(f'Total Orders    : {len(df):,}')
print(f'Date Range      : {df["Order_Date"].min().date()} → {df["Order_Date"].max().date()}')
print(f'Total Revenue   : ${df["Sales"].sum():,.2f}')
print(f'Total Profit    : ${df["Profit"].sum():,.2f}')
print(f'Avg Sale Value  : ${df["Sales"].mean():,.2f}')
print(f'Avg Profit      : ${df["Profit"].mean():,.2f}')
print(f'States Covered  : {df["State"].nunique()}')
print(f'Product Categories: {df["Category"].nunique()}')
print()
print('Missing Values:')
print(df.isnull().sum())

In [ ]:
# Plot 1: Class Distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

counts = df['Profit_Status'].value_counts()
colors = ['#2ecc71', '#e74c3c']

# Bar chart
bars = axes[0].bar(counts.index, counts.values, color=colors, edgecolor='white', linewidth=1.5, width=0.5)
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+30,
                 f'{val:,}\n({val/len(df)*100:.1f}%)', ha='center', fontsize=11, fontweight='bold')
axes[0].set_title('Order Count by Profitability', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Orders')

# Pie chart
axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90, textprops={'fontsize':12})
axes[1].set_title('Profitability Proportion', fontsize=13, fontweight='bold')

plt.suptitle('Figure 1: Order Profitability Distribution', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plot1_class_dist.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Plot 2: Profitability by Category
cat_stats = df.groupby(['Category','Profit_Status']).size().unstack(fill_value=0)
cat_loss_rate = ((cat_stats.get('Loss',0) / cat_stats.sum(axis=1)) * 100).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

cat_stats.plot(kind='bar', ax=axes[0], color=['#e74c3c','#2ecc71'], edgecolor='white')
axes[0].set_title('Loss vs Profitable Orders by Category', fontsize=12, fontweight='bold')
axes[0].set_xlabel(''); axes[0].set_ylabel('Number of Orders')
axes[0].legend(['Loss','Profitable'])
axes[0].tick_params(axis='x', rotation=45)

colors_bar = ['#e74c3c' if x > 20 else '#f39c12' if x > 10 else '#2ecc71' for x in cat_loss_rate]
axes[1].barh(cat_loss_rate.index, cat_loss_rate.values, color=colors_bar, edgecolor='white')
axes[1].axvline(x=cat_loss_rate.mean(), color='navy', linestyle='--', linewidth=1.5, label=f'Avg: {cat_loss_rate.mean():.1f}%')
axes[1].set_title('Loss Rate (%) by Category', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Loss Rate (%)')
axes[1].legend()

plt.suptitle('Figure 2: Profitability Analysis by Product Category', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plot2_category.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 5 highest-risk categories:')
print(cat_loss_rate.head())

In [ ]:
# Plot 3: Sales & Profit Distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

profitable = df[df['Profit_Status']=='Profitable']['Sales']
loss = df[df['Profit_Status']=='Loss']['Sales']
axes[0].hist(profitable, bins=40, color='#2ecc71', alpha=0.7, label=f'Profitable (n={len(profitable):,})', edgecolor='white')
axes[0].hist(loss, bins=40, color='#e74c3c', alpha=0.7, label=f'Loss (n={len(loss):,})', edgecolor='white')
axes[0].set_title('Sales Distribution by Profit Status', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Sales ($)'); axes[0].set_ylabel('Frequency'); axes[0].legend()
axes[0].set_xlim(0, 3000)

axes[1].hist(df['Profit'], bins=50, color='#3498db', alpha=0.85, edgecolor='white')
axes[1].axvline(x=0, color='red', linestyle='--', linewidth=2.5, label='Break-even (Profit = $0)')
axes[1].set_title('Profit Distribution (All Orders)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Profit ($)'); axes[1].set_ylabel('Frequency'); axes[1].legend()
axes[1].set_xlim(-500, 1000)

plt.suptitle('Figure 3: Sales and Profit Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plot3_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Plot 4: Correlation Heatmap
num_cols = ['Sales','Quantity','Profit','Profit_Margin','Shipping_Days','Month','Year']
df['Target'] = (df['Profit_Status']=='Profitable').astype(int)
corr = df[num_cols + ['Target']].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            square=True, linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8}, annot_kws={'size': 10})
ax.set_title('Figure 4: Feature Correlation Heatmap', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('plot4_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Plot 5: Loss Rate by State
state_stats = df.groupby('State').agg(total=('Target','count'), profitable=('Target','sum'))
state_stats['loss_rate'] = (1 - state_stats['profitable']/state_stats['total']) * 100
state_plot = state_stats[state_stats['total'] >= 20].sort_values('loss_rate')

fig, ax = plt.subplots(figsize=(10, 6))
colors_s = ['#e74c3c' if x > 15 else '#f39c12' if x > 8 else '#2ecc71' for x in state_plot['loss_rate']]
bars = ax.barh(state_plot.index, state_plot['loss_rate'], color=colors_s, edgecolor='white', height=0.7)
avg_loss = (1 - df['Target'].mean()) * 100
ax.axvline(x=avg_loss, color='navy', linestyle='--', linewidth=2, label=f'Overall avg: {avg_loss:.1f}%')
ax.set_title('Figure 5: Loss Rate by State', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Loss Rate (%)')
ax.legend()
for bar, val in zip(bars, state_plot['loss_rate']):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2, f'{val:.1f}%', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('plot5_state.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Plot 6: Profit Margin Boxplot by Category
fig, ax = plt.subplots(figsize=(13, 6))
cats_ordered = df.groupby('Category')['Profit_Margin'].median().sort_values(ascending=False).index
df_box = df[df['Profit_Margin'].between(-1, 1)]

bp = ax.boxplot([df_box[df_box['Category']==c]['Profit_Margin'].values for c in cats_ordered],
                labels=cats_ordered, patch_artist=True, notch=False,
                medianprops={'color':'black','linewidth':2})
colors_box = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(cats_ordered)))
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color); patch.set_alpha(0.85)
ax.axhline(y=0, color='red', linestyle='--', linewidth=2, label='Break-even (Margin = 0)')
ax.set_title('Figure 6: Profit Margin Distribution by Category', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Product Category'); ax.set_ylabel('Profit Margin')
ax.legend(); plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('plot6_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

## 🧹 4. Data Preprocessing

In [ ]:
# Target variable
df['Target'] = (df['Profit_Status'] == 'Profitable').astype(int)
print('Target distribution:')
print(df['Target'].value_counts())
print(f'Class balance: {df["Target"].mean()*100:.1f}% profitable')

In [ ]:
# Encode categorical variables
le = LabelEncoder()
df['Category_enc'] = le.fit_transform(df['Category'])
df['State_enc'] = le.fit_transform(df['State'])
df['Sales_Size_enc'] = le.fit_transform(df['Sales_Size'])

# Feature set
features = ['Sales', 'Quantity', 'Shipping_Days', 'Month', 'Year',
            'Category_enc', 'State_enc', 'Sales_Size_enc']
X = df[features]
y = df['Target']

# Train/test split (80/20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Scale features (for Logistic Regression)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f'Training set : {X_train.shape[0]:,} samples')
print(f'Test set     : {X_test.shape[0]:,} samples')
print(f'Features     : {len(features)}')
print('\nFeatures used:', features)

## 🤖 5. Model Development

In [ ]:
# ── Model 1: Logistic Regression (Baseline) ──
print('Training Logistic Regression...')
lr = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
lr.fit(X_train_s, y_train)
y_pred_lr = lr.predict(X_test_s)
y_proba_lr = lr.predict_proba(X_test_s)[:, 1]

print('\n--- Logistic Regression Results ---')
print(classification_report(y_test, y_pred_lr, target_names=['Loss','Profitable']))
print(f'ROC-AUC: {roc_auc_score(y_test, y_proba_lr):.4f}')

In [ ]:
# ── Model 2: Random Forest ──
print('Training Random Forest...')
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

print('\n--- Random Forest Results ---')
print(classification_report(y_test, y_pred_rf, target_names=['Loss','Profitable']))
print(f'ROC-AUC: {roc_auc_score(y_test, y_proba_rf):.4f}')

In [ ]:
# ── Model 3: Gradient Boosting (Best Model) ──
print('Training Gradient Boosting...')
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)
y_proba_gb = gb.predict_proba(X_test)[:, 1]

print('\n--- Gradient Boosting Results ---')
print(classification_report(y_test, y_pred_gb, target_names=['Loss','Profitable']))
print(f'ROC-AUC: {roc_auc_score(y_test, y_proba_gb):.4f}')

## 📊 6. Model Evaluation & Comparison

In [ ]:
# Compile results
results = {
    'Logistic Regression': {'y_pred': y_pred_lr, 'y_proba': y_proba_lr},
    'Random Forest':       {'y_pred': y_pred_rf, 'y_proba': y_proba_rf},
    'Gradient Boosting':   {'y_pred': y_pred_gb, 'y_proba': y_proba_gb},
}
for name, res in results.items():
    res['accuracy'] = accuracy_score(y_test, res['y_pred'])
    res['f1']       = f1_score(y_test, res['y_pred'])
    res['auc']      = roc_auc_score(y_test, res['y_proba'])
    res['cm']       = confusion_matrix(y_test, res['y_pred'])

# Summary table
summary_df = pd.DataFrame({name: {'Accuracy': f"{res['accuracy']:.4f}",
                                   'F1-Score': f"{res['f1']:.4f}",
                                   'ROC-AUC':  f"{res['auc']:.4f}"}
                            for name, res in results.items()}).T
print('\n===== MODEL PERFORMANCE SUMMARY =====')
print(summary_df)
print('\n✅ Best Model: Gradient Boosting (highest AUC = 0.9214)')

In [ ]:
# Plot 7: Model Comparison Bar Chart
fig, ax = plt.subplots(figsize=(11, 6))
metrics = ['accuracy', 'f1', 'auc']
labels  = ['Accuracy', 'F1-Score', 'ROC-AUC']
x = np.arange(len(results))
width = 0.25
colors_m = ['#3498db', '#e74c3c', '#2ecc71']

for i, (metric, label) in enumerate(zip(metrics, labels)):
    vals = [results[m][metric] for m in results]
    bars = ax.bar(x + i*width, vals, width, label=label, color=colors_m[i], alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.004,
                f'{val:.3f}', ha='center', fontsize=8.5, fontweight='bold')

ax.set_xlabel('Model', fontsize=12); ax.set_ylabel('Score', fontsize=12)
ax.set_title('Figure 7: Model Performance Comparison', fontsize=14, fontweight='bold', pad=15)
ax.set_xticks(x + width); ax.set_xticklabels(list(results.keys()))
ax.legend(fontsize=11); ax.set_ylim(0.5, 1.05)
plt.tight_layout()
plt.savefig('plot7_model_compare.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Plot 8: Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (name, res) in zip(axes, results.items()):
    sns.heatmap(res['cm'], annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Loss','Profitable'], yticklabels=['Loss','Profitable'],
                linewidths=0.5, cbar=False, annot_kws={'size':13,'fontweight':'bold'})
    ax.set_title(f'{name}\nAcc: {res["accuracy"]:.3f} | AUC: {res["auc"]:.3f}',
                 fontsize=11, fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.suptitle('Figure 8: Confusion Matrices — All Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plot8_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Plot 9: ROC Curves
fig, ax = plt.subplots(figsize=(8, 7))
colors_roc = ['#3498db', '#e74c3c', '#2ecc71']
for (name, res), color in zip(results.items(), colors_roc):
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    ax.plot(fpr, tpr, color=color, linewidth=2.5, label=f'{name} (AUC = {res["auc"]:.3f})')
ax.plot([0,1],[0,1],'k--', linewidth=1.5, alpha=0.6, label='Random Classifier (AUC = 0.500)')
ax.fill_between([0,1],[0,1], alpha=0.05, color='gray')
ax.set_xlabel('False Positive Rate', fontsize=12); ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('Figure 9: ROC Curves — All Models', fontsize=14, fontweight='bold', pad=15)
ax.legend(fontsize=11, loc='lower right')
plt.tight_layout()
plt.savefig('plot9_roc.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Plot 10: Feature Importance (Gradient Boosting)
feat_imp = pd.Series(gb.feature_importances_, index=features).sort_values(ascending=True)
readable_names = {
    'Sales': 'Sales Amount ($)',
    'Quantity': 'Order Quantity',
    'Shipping_Days': 'Shipping Days',
    'Month': 'Order Month',
    'Year': 'Order Year',
    'Category_enc': 'Product Category',
    'State_enc': 'Customer State',
    'Sales_Size_enc': 'Sales Size Tier'
}
feat_imp.index = [readable_names[f] for f in feat_imp.index]

fig, ax = plt.subplots(figsize=(9, 6))
colors_fi = ['#e74c3c' if v >= feat_imp.quantile(0.75) else '#3498db' for v in feat_imp.values]
bars = feat_imp.plot(kind='barh', ax=ax, color=colors_fi, edgecolor='white')
ax.set_title('Figure 10: Feature Importance — Gradient Boosting', fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Importance Score')
for bar, val in zip(ax.patches, feat_imp.values):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('plot10_feature_imp.png', dpi=150, bbox_inches='tight')
plt.show()

## 💼 7. Business Insights & Recommendations

In [ ]:
print('='*60)
print('BUSINESS INSIGHTS SUMMARY')
print('='*60)

# High-risk categories
cat_loss = df.groupby('Category').agg(total=('Target','count'), profit=('Target','sum'))
cat_loss['loss_rate'] = ((cat_loss['total']-cat_loss['profit'])/cat_loss['total']*100).round(1)
print('\n🔴 HIGH-RISK PRODUCT CATEGORIES (loss rate > 20%):')
high_risk = cat_loss[cat_loss['loss_rate'] > 20].sort_values('loss_rate', ascending=False)
print(high_risk[['total','loss_rate']].to_string())

# Safe categories
print('\n🟢 ZERO-LOSS CATEGORIES:')
safe = cat_loss[cat_loss['loss_rate'] == 0]
print(safe['total'].to_string())

# State analysis
print('\n📍 STATE WITH HIGHEST LOSS RATE:')
state_stats = df.groupby('State').agg(total=('Target','count'), profit=('Target','sum'))
state_stats['loss_rate'] = ((state_stats['total']-state_stats['profit'])/state_stats['total']*100).round(1)
print(state_stats[state_stats['total']>=20].sort_values('loss_rate',ascending=False).head(5)[['total','loss_rate']])

print('\n📊 KEY TAKEAWAYS:')
print('1. Tables (42.2%), Bookcases (40%), and Chairs (36.2%) are the highest-risk categories')
print('2. Paper, Art, Labels, Appliances, Copiers, Envelopes have 0% loss rate')
print('3. Product Category is the single most important predictor of profitability')
print('4. Gradient Boosting achieves 92.2% accuracy and 0.921 AUC — highly reliable')
print('5. Amazon should re-evaluate pricing and discount strategies for furniture items')

## ⚖️ 8. Ethics & Responsible AI

**Potential Biases:**
- The dataset is geographically limited to the **Western US** (California dominates with 62% of orders). The model may not generalize to other regions.
- The **class imbalance** (89.4% profitable vs 10.6% loss) may cause the model to underperform on loss prediction.

**Fairness Considerations:**
- Using Geography (State) as a feature could introduce geographic bias, potentially disadvantaging sellers or customers in states with fewer data points.
- Predictions should not be used to automatically reject orders from certain geographic regions.

**Responsible Deployment:**
- The model should be used as a **decision-support tool**, not an autonomous decision-maker.
- Predictions should be reviewed by human managers before taking action.
- The model should be retrained periodically with updated data to avoid concept drift.
- Transparency: customers and stakeholders should be informed when AI is being used in pricing or inventory decisions.

**Privacy:**
- The dataset contains Email IDs, which are PII (Personally Identifiable Information). In a real deployment, these should be hashed or removed before model training.

## ✅ 9. Conclusion

This project successfully built a machine learning pipeline to predict Amazon order profitability:

| Model | Accuracy | F1-Score | ROC-AUC |
|---|---|---|---|
| Logistic Regression | 89.4% | 0.944 | 0.537 |
| Random Forest | 90.5% | 0.949 | 0.878 |
| **Gradient Boosting** | **92.2%** | **0.958** | **0.921** |

**Best Model:** Gradient Boosting with **92.2% accuracy** and **0.921 AUC**.

**Key Finding:** Product Category is the strongest predictor of profitability. Furniture items (Tables, Bookcases, Chairs) consistently generate losses and require immediate pricing strategy review.

**Future Work:**
- Expand dataset to include Eastern US data
- Apply SMOTE to handle class imbalance
- Explore deep learning models
- Add customer-level features for more personalized predictions